## [Quest] 1. Transformer와 비교하여 변경이 필요한 부분
: GPT-1 모델은 기존 Transformer 아키텍쳐를 기반으로 하지만, Unsupervised Pre-training과 Generative Model의 특성에 맞춰 구조적 변경이 필요함.
<br>
→ 논문 4.1 Setup - Model specifications 참고

<br>
<br>

###1. 구조 : Encoder를 제거하고, Decoder만 사용하는 구조를 선택함.
기존 Transformer는 번역같은 Sequence-to-Sequence 작업을 위해 Encoder와 Decoder가 모두 존재하는 구조임.
<br>
→ GPT-1은 언어 모델링을 목표로 하기때문에, 입력 문맥을 통해 다음 토큰을 예측하는 Decoder-only Transformer 구조를 사용함. 따라서 Encoder 부분은 완전히 제거함.
<br>
<br>

###2. 블록 내부 구조: Encoder-Decoder Attention 제거
기존 Transformer Decoder는 Masked Self-Attention > Encoder-Decoder Attention > Feed Forward의 3가지 서브 레이어로 구성됨
<br>
→ 인코더가 존재하지 않으므로, 디코더 블록 내에서 인코더의 출력을 참조하는 Encoder-Decoder Multi-Head Attention(Cross-Attention) 레이어를 제거.
GPT-1의 블록은 Masked Self-Attention과 Position-wise Feed Forward 두 가지 서브 레이어로만 구성.
<br>
<br>

###3. 위치 임베딩 (Positional Embedding): 학습 가능한 임베딩 사용
기존 Transformer는 사인/코사인 함수를 이용한 고정된 위치 임베딩을 사용함.
<br>
→ GPT-1은 절대적인 위치 정보를 모델이 데이터로부터 직접 학습할 수 있도록 Learned Position Embedding을 사용.
<br>
<br>

###4. 활성화 함수 (Activation Function): GELU 적용
기존 Transformer: ReLU 함수를 사용함.
<br>
→ GPT-1은 정규화 효과와 비선형성을 동시에 고려한 GELU를 활성화 함수로 사용



In [1]:
print("-" * 245)

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


## [QUEST 평가기준] 2번: 모델의 입력 형태에 맞게 전처리 수행
챗봇 데이터(질문 Q와 답변 A)를 분리하지 않고, 하나의 긴 시퀀스로 연결한 뒤, 모델이 앞 단어들을 보고 뒷 단어를 맞추도록 데이터를 가공

In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset
import os

class ChatbotDataset(Dataset):
    def __init__(self, tokenizer, max_len=40):
        # 1. 데이터 로드
        file_path = os.path.expanduser("~/work/transformer_chatbot/data/ChatbotData.csv")
        self.data = pd.read_csv(file_path)
        self.questions = self.data['Q'].tolist()
        self.answers = self.data['A'].tolist()

        self.tokenizer = tokenizer
        self.max_len = max_len

        # 2. 특수 토큰
        self.bos_token = tokenizer.bos_token_id
        self.eos_token = tokenizer.eos_token_id
        self.pad_token = tokenizer.pad_token_id

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        # 3. 입력 시퀀스 생성: <bos> Q <eos> A <eos>
        q_tokens = self.tokenizer.encode(self.questions[idx], add_special_tokens=False)
        a_tokens = self.tokenizer.encode(self.answers[idx], add_special_tokens=False)

        input_ids = [self.bos_token] + q_tokens + [self.eos_token] + a_tokens + [self.eos_token]

        # 4. 길이 제한 및 패딩
        if len(input_ids) > self.max_len:
            input_ids = input_ids[:self.max_len]
        else:
            input_ids += [self.pad_token] * (self.max_len - len(input_ids))

        # 5. 입력과 정답 분리 (Next Token Prediction)
        input_tensor = torch.tensor(input_ids[:-1], dtype=torch.long)
        label_tensor = torch.tensor(input_ids[1:], dtype=torch.long)

        # 패딩 마스킹 (-100)
        label_tensor = torch.where(label_tensor == self.pad_token, torch.tensor(-100), label_tensor)

        return {
            "input_ids": input_tensor,
            "labels": label_tensor
        }

### [QUEST 평가기준] 3번: 모델의 입력 블록을 GPT 논문에 기반하여 수정
고정된 위치 인코딩 대신 학습 가능한 위치 임베딩(Learned Position Embedding)을 사용

In [3]:
import torch
import torch.nn as nn

class GPTInputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, dropout=0.1):
        super().__init__()
        # 1. 토큰 임베딩
        self.token_emb = nn.Embedding(vocab_size, d_model)

        # 2. 학습 가능한 위치 임베딩
        # 기존 Transformer의 Sinusoidal 대신 nn.Embedding 사용
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len = x.size()

        # 위치 인덱스 생성 (0 ~ seq_len-1)
        positions = torch.arange(seq_len, device=x.device).expand(batch_size, seq_len)

        # 입력값 + 위치정보 결합 (h0 = UWe + Wp)
        output = self.token_emb(x) + self.pos_emb(positions)

        return self.dropout(output)

### [QUEST 평가기준] 4번: GPT 모델을 정상적으로 구성

In [4]:
import torch
import torch.nn as nn
import math

# 1. 활성화 함수 (GELU) - 논문 구현
class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * torch.pow(x, 3))))

# 2. Multi-Head Attention (Masked)
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size, seq_len, _ = q.size()

        # Head 분리
        q = self.W_q(q).view(batch_size, seq_len, self.n_head, self.d_head).transpose(1, 2)
        k = self.W_k(k).view(batch_size, seq_len, self.n_head, self.d_head).transpose(1, 2)
        v = self.W_v(v).view(batch_size, seq_len, self.n_head, self.d_head).transpose(1, 2)

        # Scaled Dot-Product Attention
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)

        # Masking (Look-ahead Mask for Decoder)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)

        attn_probs = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, v)

        # Head 결합
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        return self.W_o(output)

# 3. Feed Forward Network
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.activation = GELU() # ReLU 대신 GELU 사용
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(self.activation(self.linear1(x)))

# 4. Decoder Block (Encoder 제거, Masked Self-Attention만 사용)
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_head)
        self.ffn = PositionwiseFeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Sub-layer 1: Masked Self-Attention
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))

        # Sub-layer 2: Feed Forward
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))
        return x

# 5. GPT-1 전체 모델 (Transformer Decoder-only)
class GPT1Model(nn.Module):
    def __init__(self, vocab_size, d_model=768, n_head=12, n_layers=12, max_len=512, dropout=0.1):
        super().__init__()
        # 앞서 만든 Embedding 클래스 활용
        self.embedding = GPTInputEmbedding(vocab_size, d_model, max_len, dropout)

        # 12개의 Decoder Layer 스택
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_head, d_ff=d_model*4, dropout=dropout)
            for _ in range(n_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)

    def make_causal_mask(self, x):
        # 앞으로 나올 토큰을 보지 못하게 하는 Look-ahead Mask (Triangular)
        seq_len = x.size(1)
        mask = torch.tril(torch.ones(seq_len, seq_len)).to(x.device)
        return mask

    def forward(self, x):
        mask = self.make_causal_mask(x)

        out = self.embedding(x)

        for layer in self.layers:
            out = layer(out, mask)

        return self.fc_out(out)

### [QUEST 평가기준] 5. 입력에 따른 출력이 생성
간단한 학습 루프와, 학습된 모델을 사용해 답변을 생성하는 generate_response 함수

In [6]:
import torch
import torch.optim as optim
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from transformers import PreTrainedTokenizerFast

# 1. 토크나이저 로드 (KoGPT2)
tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2",
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>')

# 2. 데이터셋 클래스
class ChatbotDataset(Dataset):
    def __init__(self, tokenizer, max_len=40):
        file_path = '/content/ChatbotData.csv'

        self.data = pd.read_csv(file_path)
        self.questions = self.data['Q'].tolist()
        self.answers = self.data['A'].tolist()

        self.tokenizer = tokenizer
        self.max_len = max_len
        self.bos_token = tokenizer.bos_token_id
        self.eos_token = tokenizer.eos_token_id
        self.pad_token = tokenizer.pad_token_id

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        q_tokens = self.tokenizer.encode(self.questions[idx], add_special_tokens=False)
        a_tokens = self.tokenizer.encode(self.answers[idx], add_special_tokens=False)

        input_ids = [self.bos_token] + q_tokens + [self.eos_token] + a_tokens + [self.eos_token]

        if len(input_ids) > self.max_len:
            input_ids = input_ids[:self.max_len]
        else:
            input_ids += [self.pad_token] * (self.max_len - len(input_ids))

        input_tensor = torch.tensor(input_ids[:-1], dtype=torch.long)
        label_tensor = torch.tensor(input_ids[1:], dtype=torch.long)

        label_tensor = torch.where(label_tensor == self.pad_token, torch.tensor(-100), label_tensor)

        return {"input_ids": input_tensor, "labels": label_tensor}

# 3. 학습 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
learning_rate = 1e-4
epochs = 10

# 모델 생성
model = GPT1Model(vocab_size=len(tokenizer), d_model=768, n_head=12, n_layers=12, dropout=0.1).to(device)

criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

dataset = ChatbotDataset(tokenizer=tokenizer, max_len=40)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# 4. 학습 루프
model.train()

for epoch in range(epochs):
    total_loss = 0.0
    for batch in dataloader:
        inputs = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.view(-1, len(tokenizer)), labels.view(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss / len(dataloader):.4f}")

Epoch 1 Loss: 5.7243
Epoch 2 Loss: 4.4326
Epoch 3 Loss: 3.8516
Epoch 4 Loss: 3.3797
Epoch 5 Loss: 2.9375
질문: 오늘 날씨 어때?
답변: 오늘 날씨 어때? 저도 좋아해요.


In [7]:
# 5. 답변 생성 테스트
def generate_response(model, tokenizer, question, max_len=40):
    model.eval()
    tokens = [tokenizer.bos_token_id] + tokenizer.encode(question, add_special_tokens=False) + [tokenizer.eos_token_id]
    input_ids = torch.tensor([tokens], dtype=torch.long).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            outputs = model(input_ids)
            next_token = torch.argmax(outputs[:, -1, :], dim=-1).unsqueeze(0)
            if next_token.item() == tokenizer.eos_token_id: break
            input_ids = torch.cat([input_ids, next_token], dim=1)

    generated_tokens = input_ids[0].tolist()
    try:
        q_end_idx = generated_tokens.index(tokenizer.eos_token_id)
        return tokenizer.decode(generated_tokens[q_end_idx+1:], skip_special_tokens=True)
    except:
        return tokenizer.decode(generated_tokens, skip_special_tokens=True)

print(f"질문: 오늘 날씨 어때?")
print(f"답변: {generate_response(model, tokenizer, '오늘 날씨 어때?')}")

질문: 오늘 날씨 어때?
답변: 오늘 날씨 어때? 저도 좋아해요.
